# Chem 427, Project #11: Nanoparticle size by three techniques

**Question.** In this project we aim to determine if AFM height reads silica nanoparticles smaller than SEM and SAXS do, while reading gold nanoparticles correctly, by more than the combined uncertainty. Additionally, we aim to determine if our own particle segmentation can reproduce the expert-annotated size distribution.

## Data availability

All data are public, were downloaded on 2026-11-08, and are committed in `data/` exactly as downloaded; the MD5 checksum of each zip matches the one listed on its Zenodo record. All cleaning happens in code below, and the notebook does not download anything.

| Files in `data/` | Content | Source | License |
|---|---|---|---|
| `VSL.zip` | VSL laboratory images as 2D arrays | pollen-metrology.com, *nPSize VSL images as 2D arrays*, Zenodo (2021), https://doi.org/10.5281/zenodo.5578927 | CC0 |
| `LNE1.zip`, `LNE2.zip`, `LNE3.zip` | LNE laboratory images as 2D arrays | pollen-metrology.com, *nPSize LNE images as 2D arrays*, Zenodo (2021), https://doi.org/10.5281/zenodo.5578878 | CC0 |
| `annotations.zip` | Expert particle annotations (`annotations.json`) | pollen-metrology.com, *nPSize JSON file: image annotations*, Zenodo (2021), https://doi.org/10.5281/zenodo.5577401 | CC0 |
| `nPSize*.dat` | SAXS curves | Deumer, J., Gollwitzer, C., Zenodo (2022), https://doi.org/10.5281/zenodo.5886834 | CC BY 4.0 |
| `17NRM04_D3_ValidationReport_v1.0_Accepted.pdf` | Validation report with reference sizes | Bartczak, D., Hodoroaba, V.-D., Zenodo (2022), https://doi.org/10.5281/zenodo.7016466 | CC BY 4.0 |

Accessed 2026-11-08.

## Setup and imports

The next cell imports the libraries used below: `json` and `zipfile` to read the annotations from inside their zip, NumPy, pandas and Matplotlib. It prints nothing when it works.

In [ ]:
import json
import sys
import zipfile

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Parameters

Every number that encodes a scientific or statistical choice is set here, with its unit and its reason. Running this cell prints nothing.

In [ ]:
ANNOTATIONS_ZIP = "data/annotations.zip"   # expert annotations, as downloaded; read from inside the zip
TECHNIQUES = ["AFM", "SEM"]                # techniques compared in the question; TEM and TSEM are not used (N4)
MATERIALS = ["01", "10"]    # nPSize01 = gold, nPSize10 = silica: the two materials in the question
GOLD_SPLIT_NM = 45          # nm; gold particles below are the 30 nm population, at or above the 60 nm one.
                            # 45 nm is the midpoint of the nominal sizes, (30 + 60) / 2, and lies in the gap between them.
RNG_SEED = 427              # dimensionless; fixed so bootstrap CIs are identical on every rerun
N_BOOTSTRAP = 10_000        # resamples per bootstrap
CI_LEVEL = 0.95             # fraction; conventional two-sided confidence level
BOOTSTRAP_UNIT = "image"    # resampling unit; particles in one image share the tip or beam, calibration and
                            # background, so they err together (S14: draw whole the unit that errs together)
HIST_BIN_WIDTH_NM = {"01": 4.0, "10": 2.0}         # nm; at least one SEM pixel step, so whole-pixel sizes do not make a comb
HIST_RANGE_NM = {"01": (0, 110), "10": (30, 70)}   # nm; covers every in-scope size
OUTLIER_TRIM_FRACTION = 0.01   # fraction dropped at each end in the outlier check only; nothing is removed (N6)

rng = np.random.default_rng(RNG_SEED)

## Load the expert annotations

`annotations.json` nests laboratory → microscope → image → particle. This cell flattens it into a tidy table, `annotations`: one row per expert-annotated particle in one image, one column per variable. The columns are the laboratory, technique, material (`npsize`), image, pixel size (nm/px), AFM height (nm, AFM rows only), minimum and maximum Feret diameter (nm) and centroid (nm). Each measurement is stored as a `{unit, value}` pair, and the cell stops if any unit is not nm. Expect the table's shape (11,227 rows, 10 columns) and its first rows.

In [ ]:
SIZE_FIELDS = {                                  # JSON key -> column name, all in nm
    "afm_height": "afm_height_nm",
    "minimum_feret_diameter": "feret_min_nm",
    "maximum_feret_diameter": "feret_max_nm",
}

with zipfile.ZipFile(ANNOTATIONS_ZIP) as archive:
    raw = json.load(archive.open("annotations.json"))

rows = []
n_skipped = 0                                    # JSON entries that are not microscopes or particles
for lab in raw.values():
    for scope_key, scope in lab.items():
        if not scope_key.startswith("microscope"):   # e.g. the lab's "name"
            n_skipped += 1
            continue
        for image in scope["data"].values():
            for particle_key, particle in image["particles"].items():
                if not particle_key.startswith("particle"):   # e.g. the image's "total"
                    n_skipped += 1
                    continue
                measurements = particle["measurements"]
                for key in [*SIZE_FIELDS, "centroid"]:
                    if measurements[key]["unit"] != "nm":
                        raise ValueError(f"{key} is in {measurements[key]['unit']}, not nm")
                row = {
                    "laboratory": lab["name"],
                    "technique": scope["type"],
                    "npsize": image["npsize"],
                    "image": image["name"],
                    "pixel_size_nm": image["pixel_size"],       # nm per pixel
                }
                for key, column in SIZE_FIELDS.items():
                    row[column] = measurements[key]["value"]
                row["centroid_x_nm"] = measurements["centroid"]["x"]
                row["centroid_y_nm"] = measurements["centroid"]["y"]
                rows.append(row)

annotations = pd.DataFrame(rows)
for column in SIZE_FIELDS.values():
    annotations[column] = pd.to_numeric(annotations[column])   # None -> NaN

print(annotations.shape)
annotations.head()

## Problems found on loading

This cell keeps the groups in the question (gold and silica, AFM and SEM) and checks them for problems that could bias a size comparison. It removes nothing. The checks: the particles, images and pixel sizes per group; missing AFM heights, where the count should equal the number of SEM rows, since only AFM measures height, with none missing on AFM rows; impossible sizes; the same particle twice (same image and centroid); and JSON entries the loader skipped. Expect one line per check.

In [ ]:
in_scope = annotations[annotations["npsize"].isin(MATERIALS) & annotations["technique"].isin(TECHNIQUES)]
print(f"{len(in_scope)} of {len(annotations)} particles in scope")
for (material, technique), rows_g in in_scope.groupby(["npsize", "technique"]):
    pixel_sizes = ", ".join(f"{p:.2f}" for p in sorted(rows_g["pixel_size_nm"].unique()))
    print(f"  nPSize{material} {technique}: {len(rows_g)} particles in {rows_g['image'].nunique()} images, "
          f"pixel size {pixel_sizes} nm/px")

is_sem = in_scope["technique"] == "SEM"
size_columns = list(SIZE_FIELDS.values())       # nm
impossible = (in_scope[size_columns] <= 0).any(axis=1) | (in_scope["feret_min_nm"] > in_scope["feret_max_nm"])
same_particle = in_scope.duplicated(subset=["image", "centroid_x_nm", "centroid_y_nm"])
print(f"Missing AFM height: {in_scope.loc[is_sem, 'afm_height_nm'].isna().sum()} on {is_sem.sum()} SEM rows, "
      f"{in_scope.loc[~is_sem, 'afm_height_nm'].isna().sum()} on AFM rows")
print(f"Impossible sizes (<= 0 nm, or min Feret > max Feret): {impossible.sum()}")
print(f"Same particle twice (same image and centroid): {same_particle.sum()}")
print(f"JSON entries skipped by the loader: {n_skipped}")

1. **AFM height exists only for AFM.** It is missing on all 605 SEM rows and on 0 of 662 AFM rows, as expected, since only AFM measures height. Kept as NaN; SEM is compared by its minimum Feret diameter.
2. **Pixel size differs between labs:** 1.40 and 1.84 nm/px for gold SEM, 0.92 and 1.86 nm/px for silica SEM, against 5.87 nm/px for all AFM. Kept, since every size is already in nm; the analysis resamples SEM within each laboratory.
3. **Silica AFM is a small group:** 42 particles in 10 images, against 287 by SEM. Kept; its uncertainty is carried by the image bootstrap.
4. **Checked, none found:** 0 impossible sizes and 0 particles counted twice.
5. **Format quirks:** 558 JSON entries that are not microscopes or particles (each laboratory's name and each image's total) are skipped by the loader.
6. **Outliers:** kept; the medians are unchanged without the extremes (N6; checked after the histogram below).

## Explore

This cell gives each particle one size, `size_nm`: the AFM height for AFM and the minimum Feret diameter for SEM (N1). It then summarises each group: AFM, SEM pooled (BAM + LNE) and SEM per lab, for each material. Median and IQR are shown because the analysis is median-based, and the mean beside the median shows skew. Gold is pooled over its two populations here. Expect an 8-row table in nm, with silica AFM near 44 nm and SEM near 48 nm.

In [ ]:
in_scope = in_scope.copy()
in_scope["size_nm"] = in_scope["feret_min_nm"]                       # nm, SEM: minimum Feret (N1)
is_afm = in_scope["technique"] == "AFM"
in_scope.loc[is_afm, "size_nm"] = in_scope.loc[is_afm, "afm_height_nm"]   # nm, AFM: height

summary_rows = []
for material in MATERIALS:
    rows_m = in_scope[in_scope["npsize"] == material]
    sem = rows_m[rows_m["technique"] == "SEM"]
    groups = {
        "AFM": rows_m[rows_m["technique"] == "AFM"],
        "SEM (BAM + LNE)": sem,
        "  SEM BAM": sem[sem["laboratory"] == "BAM"],
        "  SEM LNE": sem[sem["laboratory"] == "LNE"],
    }
    for label, rows_g in groups.items():
        q1, median, q3 = rows_g["size_nm"].quantile([0.25, 0.50, 0.75])
        summary_rows.append({"material": f"nPSize{material}", "group": label,
                             "n particles": len(rows_g), "n images": rows_g["image"].nunique(),
                             "median": median, "IQR": q3 - q1, "mean": rows_g["size_nm"].mean()})

pd.DataFrame(summary_rows).set_index(["material", "group"]).round(1)   # sizes in nm

*Your sentence giving the numbers a reader should take from the table.*

This figure compares the shape of the AFM and SEM size distributions, one panel per material. The curves are probability densities, because the groups differ in size (42 to 620 particles), and each series has its own line style and marker. The dotted line on the gold panel is `GOLD_SPLIT_NM`. Expect two modes near 30 and 60 nm for gold, and one narrow peak near 45 to 50 nm for silica.

In [ ]:
SERIES_STYLE = {"AFM": ("#2a78d6", "-", "o"), "SEM": ("#eb6834", "--", "s")}   # technique -> colour, line, marker

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, material in zip(axes, MATERIALS):
    rows_m = in_scope[in_scope["npsize"] == material]
    low, high = HIST_RANGE_NM[material]
    bin_edges = np.arange(low, high + HIST_BIN_WIDTH_NM[material], HIST_BIN_WIDTH_NM[material])   # nm
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2                                            # nm
    for technique, (colour, linestyle, marker) in SERIES_STYLE.items():
        sizes = rows_m.loc[rows_m["technique"] == technique, "size_nm"]
        density, _ = np.histogram(sizes, bins=bin_edges, density=True)                           # 1/nm
        ax.plot(bin_centers, density, color=colour, linestyle=linestyle, marker=marker,
                markersize=5, label=f"{technique} (n = {len(sizes)})")
    if material == "01":
        ax.axvline(GOLD_SPLIT_NM, color="0.4", linestyle=":", label=f"GOLD_SPLIT_NM = {GOLD_SPLIT_NM} nm")
    ax.set_title(f"nPSize{material}")
    ax.set_xlabel("Particle size (nm): AFM height or SEM minimum Feret")
    ax.set_ylabel("Probability density (1/nm)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

*Your one-sentence description of the figure, with the numbers.*

This cell checks whether the most extreme particles could move a result. It assigns each particle to its population (silica, or gold below or at/above `GOLD_SPLIT_NM`), then prints each group's median with all particles and with `OUTLIER_TRIM_FRACTION`, and at least one particle, dropped at each end, plus its lowest and highest size. No particle is removed (N6). Expect medians that are unchanged at 0.1 nm.

In [ ]:
in_scope["population"] = "silica"
is_gold = in_scope["npsize"] == "01"
in_scope.loc[is_gold & (in_scope["size_nm"] < GOLD_SPLIT_NM), "population"] = "gold 30 nm"
in_scope.loc[is_gold & (in_scope["size_nm"] >= GOLD_SPLIT_NM), "population"] = "gold 60 nm"

for (population, technique), rows_g in in_scope.groupby(["population", "technique"]):
    sizes = np.sort(rows_g["size_nm"].to_numpy())                         # nm
    n_trim = max(1, int(OUTLIER_TRIM_FRACTION * len(sizes)))
    trimmed = sizes[n_trim:len(sizes) - n_trim]
    print(f"{population} {technique}: median {np.median(sizes):.1f} nm, {np.median(trimmed):.1f} nm without "
          f"{n_trim} at each end; range {sizes[0]:.1f} to {sizes[-1]:.1f} nm (n = {len(sizes)})")

## Part A: does AFM read silica smaller, and gold correctly?

*Question parts 1–4: AFM against SEM and SAXS, and against the combined uncertainty.*

### A1. Silica: AFM − SEM

*To do (draft): median difference with an image-bootstrap CI, as a figure.*

### A2. Gold: AFM − SEM per population

*To do (draft): the same for the 30 nm and 60 nm populations, split at `GOLD_SPLIT_NM`.*

### A3. SAXS

*To do (final): the report's SAXS size for each material, compared with AFM.*

### A4. Combined uncertainty

*To do (final): sampling plus calibration uncertainty, combined as √(u₁² + u₂²); is each difference larger?*

## Part B: can my own segmentation reproduce the experts?

*Question part 5.*

### B1. Load the AFM height maps

*To do (final): each map is a 2048-byte header and a 512 × 512 array of heights in nm.*

### B2. Flatten and segment

*To do (final): background plane (S11), threshold above the noise (S12), label particles.*

### B3. Particle heights

*To do (final): each particle's maximum height; median with an image-bootstrap CI.*

### B4. Compare with the experts

*To do (final): my heights against the experts' for the same material.*

## Next steps

*To do (draft): what these data cannot answer, and the assumption checked next and how.*

## Conclusion

*To do (final): the answer to each part of the question, with uncertainty and limits.*

## Environment

This cell prints the Python version and the versions of the packages used above, so the results can be reproduced. Expect Python 3.14 and the versions locked in `uv.lock`.

In [ ]:
print(f"Python {sys.version.split()[0]}")
for module in (np, pd, matplotlib):
    print(f"{module.__name__} {module.__version__}")